In [2]:
from pathlib import Path
import pandas as pd

PROJECT = Path.cwd().parent  # one level up from Silver
INPUT_DIR = PROJECT / "Bronze"
OUTPUT_DIR = Path.cwd()  # Silver itself, since the script runs from here

df = pd.read_csv(INPUT_DIR / "new_car_registrations_FI.csv", encoding="latin-1")

# Keep only the region-level summary rows (MKxx ...). MAINLAND FINLAND is excluded,
# since the country total can be derived by summing all regions - keeping it would
# double-count the totals.
is_summary_row = df["Area"].str.startswith("MK")
df_summary = df[is_summary_row].copy()

# Strip the MK code from the region label: "MK01 Uusimaa" -> "Uusimaa"
df_summary["region"] = df_summary["Area"].str.split(" ", n=1).str[1]

# Drop "Total" rows entirely
df_summary = df_summary[df_summary["Driving power"] != "Total"].copy()

# Map source Driving power values to standardized silver-level categories
DRIVING_POWER_MAP = {
    "Petrol": "petrol",
    "Diesel": "diesel",
    "Electricity": "electricity",
    "Natural gas (CNG)": "gas/gas flex",
    "Petrol/CNG": "gas/gas flex",
    "Petrol/Electricity (plug-in hybrid)": "plug-in hybrid",
    "Diesel/Electricity (plug-in hybrid)": "plug-in hybrid",
    "Petrol/Ethanol": "petrol/ethanol",
    "Other": "other",
}
df_summary["driving_power"] = df_summary["Driving power"].map(DRIVING_POWER_MAP)

# Split the Month column: "2016M01" -> year=2016, month=01
df_summary["year"] = df_summary["Month"].str.split("M").str[0]
df_summary["month"] = df_summary["Month"].str.split("M").str[1]

df_summary["country"] = "Finland"

df_lookup = df_summary.rename(columns={
    "First registrations of passenger cars": "number_of_new_registrations",
})[[
    "country",
    "year",
    "month",
    "region",
    "number_of_new_registrations",
    "driving_power",
    "fetch_timestamp",
]]

# Convert types for consistency before saving
df_lookup["year"] = df_lookup["year"].astype(int)
df_lookup["month"] = df_lookup["month"].astype(int)
df_lookup["number_of_new_registrations"] = df_lookup["number_of_new_registrations"].astype(int)
df_lookup["fetch_timestamp"] = pd.to_datetime(df_lookup["fetch_timestamp"])

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df_lookup.to_csv(OUTPUT_DIR / "silver_new_car_registrations_totals_lookup.csv", index=False)